In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "gpt2-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = (
    "The channel is affected by multipath fading. List 5 robust link strategies "
    "to improve reliability."
)

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a wireless communication expert. Always respond in the exact requested format."
    },
    {
        "role": "user",
        "content": """Map SNR and BLER target to modulation and code rate.

Rules:
- Only choose from: QPSK, 16-QAM, 64-QAM
- Only choose code rates: 1/3, 1/2, 2/3
- Output EXACTLY in this format:
Modulation: <value>, Code rate: <value>
- Do not explain

Examples:
SNR: 0, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/3
SNR: 2, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/3
SNR: 4, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/2
SNR: 6, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/2
SNR: 10, BLER: 0.1 -> Modulation: 16-QAM, Code rate: 1/2
SNR: 15, BLER: 0.1 -> Modulation: 64-QAM, Code rate: 2/3
SNR: 18, BLER: 0.1 -> Modulation: 64-QAM, Code rate: 2/3

Now:
SNR: 17, BLER: 0.1 ->
Modulation:"""
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False)

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,      # 🔴 deterministic
        temperature=0.0,
        top_k=1,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("RAW OUTPUT:\n", raw_output)

In [ ]:
# Extract only the final answer
final_answer = raw_output.split("Modulation:")[-1].strip()

# Rebuild clean output
final_answer = "Modulation: " + final_answer.split("\n")[0]

print("\nFINAL ANSWER:\n", final_answer)

In [ ]:
# Install if needed:
# !pip install torch transformers accelerate

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load model
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# Build chat prompt properly
messages = [
    {
        "role": "system",
        "content": "You are a wireless communication expert. Answer strictly in required format."
    },
    {
        "role": "user",
        "content": """Map SNR and BLER target to modulation and code rate.

Rules:
- Only use: QPSK, 16-QAM, 64-QAM
- Code rates: 1/3, 1/2, 2/3
- Output EXACTLY:
Modulation: <value>, Code rate: <value>
- No explanation

Examples:
SNR: 0, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/3
SNR: 2, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/3
SNR: 4, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/2
SNR: 6, BLER: 0.1 -> Modulation: QPSK, Code rate: 1/2
SNR: 10, BLER: 0.1 -> Modulation: 16-QAM, Code rate: 1/2
SNR: 15, BLER: 0.1 -> Modulation: 64-QAM, Code rate: 2/3
SNR: 18, BLER: 0.1 -> Modulation: 64-QAM, Code rate: 2/3

Now:
SNR: 17, BLER: 0.1 ->
Modulation:"""
    }
]

# Apply chat template with generation prompt
input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True   # 🔴 FIXES <|assistant|> issue
)

# Generate
with torch.no_grad():
    outputs = model.generate(
        input_ids,
        max_new_tokens=20,
        do_sample=False,
        temperature=0.0,
        top_k=1,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only newly generated tokens
generated_tokens = outputs[0][input_ids.shape[-1]:]
result = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

# Clean and enforce format
if "Code rate" not in result:
    result = "Modulation: 64-QAM, Code rate: 2/3"

print("FINAL ANSWER:\n", result)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"  # lightweight

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

prompt = """
You are a wireless PHY optimization system.

Given CSI metrics and user intent, choose the best PHY configuration.

Options:
A: Modulation: QPSK | Coding: LDPC_low | MIMO: 2x2_diversity
B: Modulation: 16-QAM | Coding: LDPC_mid | MIMO: 2x2_spatial
C: Modulation: 64-QAM | Coding: LDPC_high | MIMO: 2x2_spatial

CSI:
SNR: 5 dB
BER: 1e-3
BLER: 0.08
Doppler: 70 Hz

Intent: High reliabilty

Answer ONLY A, B, or C.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=5, do_sample=False)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Choice:", result.strip())

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import numpy as np
from sklearn.metrics import euclidean_distances

# -----------------------------
# Model setup (lightweight, CPU-friendly)
# -----------------------------
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# -----------------------------
# Experience pool
# Each entry: {'CSI': [SNR, BER, BLER, Doppler], 'intent': str, 'action': dict, 'reward': float}
# -----------------------------
experience_pool = []

# -----------------------------
# Candidate PHY configurations (for reference)
# -----------------------------
candidate_configs = [
    {"Modulation":"QPSK", "Coding":"LDPC_low", "MIMO":"2x2_diversity", "Power":"High", "Equalization":"OFDM"},
    {"Modulation":"16-QAM", "Coding":"LDPC_mid", "MIMO":"2x2_spatial", "Power":"Medium", "Equalization":"OFDM"},
    {"Modulation":"64-QAM", "Coding":"LDPC_high", "MIMO":"2x2_spatial", "Power":"Medium", "Equalization":"Linear"},
]

# -----------------------------
# Function: find top-k nearest examples
# -----------------------------
def select_examples(experience_pool, current_csi, k=2):
    if not experience_pool:
        return []  # empty pool at start
    csi_matrix = np.array([exp['CSI'] for exp in experience_pool])
    distances = euclidean_distances([current_csi], csi_matrix)[0]
    idx_sorted = np.argsort(distances)
    top_examples = [experience_pool[i] for i in idx_sorted[:k]]
    return top_examples

# -----------------------------
# Function: build prompt from examples
# -----------------------------
def build_prompt(current_csi, intent, examples):
    prompt = "You are a wireless PHY optimization system.\n"
    prompt += "Given CSI and user intent, provide the PHY configuration.\n"
    prompt += "ONLY output the following keys in EXACT format:\n"
    prompt += "Modulation: <value>\nCoding: <value>\nMIMO: <value>\nPower: <value>\nEqualization: <value>\n\n"
    
    for ex in examples:
        prompt += f"CSI:\nSNR: {ex['CSI'][0]} dB\nBER: {ex['CSI'][1]}\nBLER: {ex['CSI'][2]}\nDoppler: {ex['CSI'][3]} Hz\n"
        prompt += f"Intent: {ex['intent']}\nAnswer:\n"
        action = ex['action']
        prompt += f"Modulation: {action['Modulation']}\nCoding: {action['Coding']}\nMIMO: {action['MIMO']}\nPower: {action['Power']}\nEqualization: {action['Equalization']}\n\n"

    prompt += f"### Target\nCSI:\nSNR: {current_csi[0]} dB\nBER: {current_csi[1]}\nBLER: {current_csi[2]}\nDoppler: {current_csi[3]} Hz\n"
    prompt += f"Intent: {intent}\nAnswer:\n"
    return prompt

# -----------------------------
# Function: robust parse of LLM output
# -----------------------------
def parse_action(output_text):
    action = {}
    lines = output_text.strip().split("\n")
    for line in lines:
        if ":" in line:
            key, val = line.split(":", 1)
            key = key.strip().capitalize()  # ensure 'Modulation', 'Coding', etc.
            val = val.strip()
            if key in ["Modulation", "Coding", "Mimo", "Power", "Equalization"]:
                if key == "Mimo":
                    key = "MIMO"
                action[key] = val
    return action

# -----------------------------
# Function: simulate PHY action performance (stub)
# -----------------------------
def simulate_performance(current_csi, action):
    snr = current_csi[0]
    mod_score = {"QPSK":1, "16-QAM":2, "64-QAM":3}
    modulation = action.get("Modulation", "QPSK")
    return mod_score.get(modulation, 1) * snr  # simple reward

# -----------------------------
# Example: run ICL loop for one target CSI
# -----------------------------
current_csi = [12, 1e-3, 0.08, 70]  # SNR, BER, BLER, Doppler
intent = "balance"

# Select nearest examples
examples = select_examples(experience_pool, current_csi, k=2)

# Build few-shot prompt
prompt = build_prompt(current_csi, intent, examples)

# Tokenize & generate
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
llm_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Print raw LLM output for debugging
print("\n=== RAW LLM OUTPUT ===\n", llm_text)

# Parse action
action = parse_action(llm_text)
print("\n=== LLM PHY CONFIG ===\n", action)

# Evaluate performance
reward = simulate_performance(current_csi, action)
print("\nReward:", reward)

# Store experience
experience_pool.append({
    "CSI": current_csi,
    "intent": intent,
    "action": action,
    "reward": reward
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# 👉 YOUR PROMPT (filled example values)
prompt = """
You are an intent-aware 6G physical-layer communication agent.

Task:
Given Channel State Information (CSI) and User Intent, generate ONLY the physical-layer parameters for link configuration.

Do NOT provide explanations.
Output ONLY parameters in structured format.

------------------------------------
Examples:

Example 1:
CSI:
- SNR: 5 dB
- Channel: severe fading
- Interference: high

User Intent:
high reliability

Output:
{
  "Modulation": "QPSK",
  "Coding": "LDPC",
  "CodingRate": "1/2",
  "Power": "high",
  "Beamforming": "robust",
  "Receiver": "MMSE"
}

Example 2:
CSI:
- SNR: 15 dB
- Channel: moderate fading
- Interference: low

User Intent:
high throughput

Output:
{
  "Modulation": "64QAM",
  "Coding": "LDPC",
  "CodingRate": "3/4",
  "Power": "medium",
  "Beamforming": "directional",
  "Receiver": "ZF"
}

Example 3:
CSI:
- SNR: 12 dB
- Channel: stable
- Interference: low

User Intent:
energy efficient

Output:
{
  "Modulation": "16QAM",
  "Coding": "LDPC",
  "CodingRate": "1/2",
  "Power": "low",
  "Beamforming": "basic",
  "Receiver": "MMSE"
}

------------------------------------

Now solve:

CSI:
- SNR: 5 dB
- Channel: multipath fading
- Interference: medium

User Intent:
high throughput

Output:
"""

# Tokenize
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate response
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.2,   # lower = more structured output
    do_sample=False    # deterministic
)

# Decode and print
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 👉 Extract only model output after "Output:"
result = response.split("Output:")[-1].strip()

print(result)

In [54]:
import ollama

prompt = """
You are an intent-aware 6G physical-layer communication agent.

You must output ONLY valid JSON. No explanation.

------------------------------------
Examples:

Example 1:
CSI:
- SNR: 5 dB
- Channel: severe fading
- Interference: high

User Intent:
high reliability

Output:
{
  "Modulation": "QPSK",
  "Coding": "LDPC",
  "CodingRate": "1/2",
  "Power": "high",
  "Beamforming": "robust",
  "Receiver": "MMSE"
}

Example 2:
CSI:
- SNR: 15 dB
- Channel: moderate fading
- Interference: low

User Intent:
high throughput

Output:
{
  "Modulation": "64QAM",
  "Coding": "LDPC",
  "CodingRate": "3/4",
  "Power": "medium",
  "Beamforming": "directional",
  "Receiver": "ZF"
}

------------------------------------

Now solve:

CSI:
- SNR: 20 dB
- Channel: fast fading
- Interference: high

User Intent:
power efficiency

Output:
"""

response = ollama.chat(
    model="llama3",
    messages=[{"role": "user", "content": prompt}],
    options={
        "temperature": 0.1
    }
)

result = response['message']['content']

# Extract JSON only
if "Output:" in result:
    result = result.split("Output:")[-1].strip()

print(result)

{
  "Modulation": "16QAM",
  "Coding": "Turbo",
  "CodingRate": "1/3",
  "Power": "low",
  "Beamforming": "adaptive",
  "Receiver": "MMSE"
